# 02. Datasets fold-local para clasificación

Esta notebook prepara los cinco pares `train/validation` del conjunto de desarrollo ISIC sin fuga de información. Consume el manifiesto canónico; no vuelve a crear los splits.

Para cada fold ajusta **solo con su train**:

1. conversión de infinitos a faltantes,
2. imputación por mediana,
3. winsorización p1–p99, y
4. filtro iterativo de correlación de Pearson `|r| > 0.95`.

La metadata clínica se conserva cruda en cada Parquet; su imputación y codificación pertenecen al pipeline del modelo. El test congelado se exporta crudo: se transformará una sola vez cuando el pipeline ganador se reajuste sobre todo el conjunto de desarrollo.

## Contrato de seguridad

- La extracción radiomics debe haber terminado y cubrir todo el manifiesto de clasificación.
- Los estados fallidos permanecen visibles en el reporte de exclusiones; nunca desaparecen por un `inner join`.
- Los identificadores, `target` y metadatos clínicos se preservan; solo las columnas radómicas entran al transformador.
- Los subconjuntos de características pueden diferir entre folds.
- Esta etapa no escala ni ejecuta ANOVA, L1 o RFE; esas operaciones pertenecen al pipeline del modelo.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = (Path.cwd() / "../..").resolve()
if not (ROOT / "src").exists():
    raise RuntimeError("No se pudo localizar la raíz del repositorio.")
sys.path.insert(0, str(ROOT / "src"))

from scd_ml.classification import (
    DEFAULT_METADATA_COLUMNS,
    FoldLocalRadiomicsTransformer,
    build_classification_cohort,
    split_development_fold,
)
from scd_ml.features.radiomics_contract import validate_radiomics_contract

FEATURES_CSV = ROOT / "results/features/radiomics_features.csv"
STATUS_CSV = ROOT / "results/features/radiomics_status.csv"
MASKS_MANIFEST_CSV = ROOT / "results/segmentation/isic_masks_manifest.csv"
CLASSIFICATION_MANIFEST_CSV = ROOT / "results/splits/isic_classification.csv"
METADATA_PARQUET = ROOT / "results/processed/isic_model_input_raw.parquet"
OUTPUT_ROOT = ROOT / "results/classification/fold_datasets"

WINSOR_LOW = 0.01
WINSOR_HIGH = 0.99
CORRELATION_THRESHOLD = 0.95
WRITE_ARTIFACTS = True
OVERWRITE = True

print(f"Raíz: {ROOT}")
print(f"Salida: {OUTPUT_ROOT}")

Raíz: /Users/indalecio.rios/Documents/personal-github/scd-ml
Salida: /Users/indalecio.rios/Documents/personal-github/scd-ml/results/classification/fold_datasets


## 1. Validación del handoff y construcción de la cohorte

La celda se detiene si segmentación/radiomics siguen incompletos respecto al manifiesto. Esto evita entrenar accidentalmente con una cohorte parcial.

In [2]:
required_paths = [
    FEATURES_CSV,
    STATUS_CSV,
    MASKS_MANIFEST_CSV,
    CLASSIFICATION_MANIFEST_CSV,
    METADATA_PARQUET,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Faltan entradas requeridas: {missing_paths}")

handoff_summary = validate_radiomics_contract(
    MASKS_MANIFEST_CSV, FEATURES_CSV, STATUS_CSV
)
manifest = pd.read_csv(
    CLASSIFICATION_MANIFEST_CSV, dtype={"image_id": "string"}
)
features = pd.read_csv(FEATURES_CSV, dtype={"image_id": "string"})
statuses = pd.read_csv(STATUS_CSV, dtype={"image_id": "string"})
metadata = pd.read_parquet(METADATA_PARQUET)

cohort = build_classification_cohort(
    manifest, features, statuses, metadata=metadata
)
metadata_columns = list(DEFAULT_METADATA_COLUMNS)
radiomic_columns = [column for column in features.columns if column != "image_id"]
if not radiomic_columns:
    raise ValueError("No se encontraron características radiómicas.")

print(pd.Series(handoff_summary, name="value").to_string())
print(f"\nCohorte: {len(cohort):,} imágenes")
print(f"Características radiómicas crudas: {len(radiomic_columns):,}")
print(f"Metadatos clínicos crudos: {metadata_columns}")
pd.crosstab(
    [cohort["split"], cohort["target"]],
    cohort["radiomics_status"],
    margins=True,
)

expected_images      76974
successful_images    74743
failed_images         2231
feature_columns        408

Cohorte: 76,974 imágenes
Características radiómicas crudas: 408
Metadatos clínicos crudos: ['age_approx', 'anatom_site_1', 'pixels_x', 'pixels_y', 'sex']


radiomics_status                 empty_mask     ok  upstream_error    All
split target                                                             
test  Benign-melanocytic                154   9632               9   9795
      Benign-non-melanocytic            242   1675               0   1917
      Malignant-melanocytic              25   2105               4   2134
      Malignant-non-melanocytic          27   1521               1   1549
train Benign-melanocytic                731  38430              18  39179
      Benign-non-melanocytic            663   7000               5   7668
      Malignant-melanocytic              92   8427              16   8535
      Malignant-non-melanocytic         223   5953              21   6197
All                                    2157  74743              74  76974

## 2. Salidas

Cada directorio `fold_N/` contiene:

- `train.parquet`: radiomics transformado con estadísticas propias y metadata cruda;
- `validation.parquet`: radiomics transformado con las estadísticas de ese train y metadata cruda;
- `preprocessing_parameters.csv`: medianas y límites p1/p99;
- `dropped_features.csv`: variables eliminadas y motivo.

`test_raw.parquet` conserva radiomics y metadata sin transformar. `cohort_coverage_by_class.csv` documenta la cobertura por clase. `OVERWRITE=False` evita reemplazar resultados accidentalmente.

In [3]:
if WRITE_ARTIFACTS and OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.rglob("*")) and not OVERWRITE:
    raise FileExistsError(
        f"{OUTPUT_ROOT} ya contiene artefactos. Usa OVERWRITE=True para reemplazarlos."
    )

id_columns = ["image_id", "group_id", "target", "split", "cv_fold"]
selected_by_fold = {}
fold_summaries = []
parameter_reports = []
drop_reports = []

for fold in range(5):
    train_rows, validation_rows = split_development_fold(cohort, fold)
    transformer = FoldLocalRadiomicsTransformer(
        winsor_low=WINSOR_LOW,
        winsor_high=WINSOR_HIGH,
        correlation_threshold=CORRELATION_THRESHOLD,
    )
    X_train = transformer.fit_transform(train_rows[radiomic_columns])
    X_validation = transformer.transform(validation_rows[radiomic_columns])
    selected_by_fold[fold] = set(transformer.get_feature_names_out())

    train_dataset = pd.concat(
        [
            train_rows[id_columns + metadata_columns].reset_index(drop=True),
            X_train.reset_index(drop=True),
        ],
        axis=1,
    )
    validation_dataset = pd.concat(
        [
            validation_rows[id_columns + metadata_columns].reset_index(drop=True),
            X_validation.reset_index(drop=True),
        ],
        axis=1,
    )
    assert list(X_train.columns) == list(X_validation.columns)
    pd.testing.assert_frame_equal(
        train_dataset[metadata_columns],
        train_rows[metadata_columns].reset_index(drop=True),
    )
    pd.testing.assert_frame_equal(
        validation_dataset[metadata_columns],
        validation_rows[metadata_columns].reset_index(drop=True),
    )
    assert not (set(train_dataset["group_id"]) & set(validation_dataset["group_id"]))

    parameters = transformer.feature_parameters_.copy()
    parameters.insert(0, "fold", fold)
    dropped = transformer.dropped_features_.copy()
    dropped.insert(0, "fold", fold)
    parameter_reports.append(parameters)
    drop_reports.append(dropped)
    fold_summaries.append(
        {
            "fold": fold,
            "n_train": len(train_dataset),
            "n_validation": len(validation_dataset),
            "n_input_features": len(radiomic_columns),
            "n_metadata_features": len(metadata_columns),
            "n_selected_features": X_train.shape[1],
            "n_dropped_features": len(radiomic_columns) - X_train.shape[1],
        }
    )

    if WRITE_ARTIFACTS:
        fold_dir = OUTPUT_ROOT / f"fold_{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        train_dataset.to_parquet(fold_dir / "train.parquet", index=False)
        validation_dataset.to_parquet(fold_dir / "validation.parquet", index=False)
        parameters.to_csv(fold_dir / "preprocessing_parameters.csv", index=False)
        dropped.to_csv(fold_dir / "dropped_features.csv", index=False)

fold_summary = pd.DataFrame(fold_summaries)
fold_summary

,fold,n_train,n_validation,n_input_features,n_metadata_features,n_selected_features,n_dropped_features
0,0,47863,11947,408,5,137,271
1,1,47816,11994,408,5,138,270
2,2,47834,11976,408,5,138,270
3,3,47848,11962,408,5,138,270
4,4,47879,11931,408,5,137,271


## 3. Estabilidad del filtro entre folds

Que una variable cambie de estado entre folds no es fuga ni un error. Este reporte permite detectar grupos de variables redundantes cuya representación es inestable.

In [4]:
selection_frequency = pd.DataFrame(
    {
        "feature": radiomic_columns,
        "n_folds_selected": [
            sum(feature in selected_by_fold[fold] for fold in range(5))
            for feature in radiomic_columns
        ],
    }
)
selection_frequency["selection_frequency"] = selection_frequency["n_folds_selected"] / 5
selection_frequency = selection_frequency.sort_values(
    ["n_folds_selected", "feature"], ascending=[False, True]
).reset_index(drop=True)

jaccard = pd.DataFrame(index=range(5), columns=range(5), dtype=float)
for left in range(5):
    for right in range(5):
        union = selected_by_fold[left] | selected_by_fold[right]
        intersection = selected_by_fold[left] & selected_by_fold[right]
        jaccard.loc[left, right] = len(intersection) / len(union)
jaccard.index.name = "fold"
jaccard.columns.name = "fold"

coverage_by_class = (
    cohort.groupby(["split", "target"], observed=True)
    .agg(
        total_images=("image_id", "size"),
        eligible_images=("eligible_for_classification", "sum"),
    )
    .reset_index()
)
coverage_by_class["eligible_images"] = coverage_by_class["eligible_images"].astype(int)
coverage_by_class["excluded_images"] = (
    coverage_by_class["total_images"] - coverage_by_class["eligible_images"]
)
coverage_by_class["exclusion_pct"] = (
    100 * coverage_by_class["excluded_images"] / coverage_by_class["total_images"]
).round(2)

eligible_test = cohort[
    cohort["eligible_for_classification"] & cohort["split"].eq("test")
][id_columns + metadata_columns + radiomic_columns].copy()
exclusions = cohort.loc[
    ~cohort["eligible_for_classification"],
    id_columns + ["radiomics_status", "radiomics_error"],
].copy()

if WRITE_ARTIFACTS:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    eligible_test.to_parquet(OUTPUT_ROOT / "test_raw.parquet", index=False)
    exclusions.to_csv(OUTPUT_ROOT / "cohort_exclusions.csv", index=False)
    coverage_by_class.to_csv(
        OUTPUT_ROOT / "cohort_coverage_by_class.csv", index=False
    )
    fold_summary.to_csv(OUTPUT_ROOT / "fold_summary.csv", index=False)
    pd.concat(parameter_reports, ignore_index=True).to_csv(
        OUTPUT_ROOT / "preprocessing_parameters_all_folds.csv", index=False
    )
    pd.concat(drop_reports, ignore_index=True).to_csv(
        OUTPUT_ROOT / "dropped_features_all_folds.csv", index=False
    )
    selection_frequency.to_csv(
        OUTPUT_ROOT / "feature_selection_frequency.csv", index=False
    )
    jaccard.to_csv(OUTPUT_ROOT / "feature_selection_jaccard.csv")

print(f"Test crudo elegible: {len(eligible_test):,} imágenes")
print(f"Exclusiones documentadas: {len(exclusions):,}")
print("\nCobertura por clase (% de exclusión):")
display(coverage_by_class)
print("\nJaccard de características retenidas:")
display(jaccard.round(3))
selection_frequency["n_folds_selected"].value_counts().sort_index(ascending=False)

Test crudo elegible: 14,933 imágenes
Exclusiones documentadas: 2,231

Cobertura por clase (% de exclusión):


,split,target,total_images,eligible_images,excluded_images,exclusion_pct
0,test,Benign-melanocytic,9795,9632,163,1.66
1,test,Benign-non-melanocytic,1917,1675,242,12.62
2,test,Malignant-melanocytic,2134,2105,29,1.36
3,test,Malignant-non-melanocytic,1549,1521,28,1.81
4,train,Benign-melanocytic,39179,38430,749,1.91
5,train,Benign-non-melanocytic,7668,7000,668,8.71
6,train,Malignant-melanocytic,8535,8427,108,1.27
7,train,Malignant-non-melanocytic,6197,5953,244,3.94



Jaccard de características retenidas:


fold,0,1,2,3,4
fold,,,,,
0,1.000,0.978,0.950,0.923,0.916
1,0.978,1.000,0.944,0.917,0.910
2,0.950,0.944,1.000,0.930,0.910
3,0.923,0.917,0.930,1.000,0.910
4,0.916,0.910,0.910,0.910,1.000


n_folds_selected
5    126
4      9
3      3
2      2
1      9
0    259
Name: count, dtype: int64

## Siguiente paso

La notebook de modelado consumirá cada par `fold_N/train.parquet` y `fold_N/validation.parquet`. Un `ColumnTransformer` ajustará imputación/escalado de metadata numérica y la imputación/OneHotEncoder de metadata categórica exclusivamente con train. Escalamiento radómico, selección supervisada (ninguna, ANOVA F, embebida o RFE), balanceo y modelo seguirán la misma regla.

Después de elegir la configuración ganadora se reajustará el pipeline completo sobre todo development y recién entonces se transformará `test_raw.parquet`.